In [128]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import roc_auc_score, auc
from sklearn.neighbors import BallTree
from collections import defaultdict
import matplotlib.pyplot as plt
from collections import Counter

In [129]:
df = pd.read_csv("../scr/data/cleaned_rat_sightings_data/daily_zip_rs.csv")
df

,created_date,zip,count
0,2020-01-01,10026.0,1
1,2020-01-01,10031.0,1
2,2020-01-01,10035.0,1
3,2020-01-01,10128.0,1
4,2020-01-01,10314.0,1
...,...,...,...
92213,2025-12-30,11225.0,1
92214,2025-12-30,11226.0,1
92215,2025-12-30,11229.0,1
92216,2025-12-30,11237.0,1


Run the code block below to set counts to 1 to one-hot encode.

In [130]:
df['count']=1

In [131]:
df["zip"] = df["zip"].astype("Int64").astype(str).str.zfill(5)
df["count"] = df["count"].fillna(0).astype(int)
df["created_date"] = pd.to_datetime(df["created_date"])

In [132]:
# Make a dictionary which associates to each ZIP its neighbors. This is purely by border.

zip_coords = pd.read_csv("../scr/cleaning/map_data_for_cleaning/uszips.csv", dtype={"zip": str})
zip_coords["zip"] = zip_coords["zip"].str.zfill(5)
zip_coords = zip_coords[zip_coords["zip"].isin(df["zip"].unique())].reset_index(drop=True)

# Convert to radians
zip_coords["lat_rad"] = np.radians(zip_coords["lat"])
zip_coords["lon_rad"] = np.radians(zip_coords["lng"])

# BallTree for 5 nearest neighbors
tree = BallTree(zip_coords[["lat_rad","lon_rad"]].values, metric="haversine")
_, indices = tree.query(zip_coords[["lat_rad","lon_rad"]].values, k=6)

neighbor_map = {
    z: zip_coords["zip"].iloc[inds[1:]].tolist()
    for z, inds in zip(zip_coords["zip"], indices)
}

neighbor_map

{'10001': ['10018', '10011', '10036', '10020', '10016'],
 '10002': ['10009', '10012', '10278', '10038', '10013'],
 '10003': ['10010', '10012', '10009', '10014', '10011'],
 '10004': ['10005', '10280', '10006', '10038', '11231'],
 '10005': ['10006', '10038', '10280', '10007', '10278'],
 '10006': ['10280', '10005', '10007', '10282', '10038'],
 '10007': ['10278', '10006', '10282', '10038', '10013'],
 '10009': ['10003', '10002', '10010', '10012', '10016'],
 '10010': ['10016', '10003', '10009', '10011', '10017'],
 '10011': ['10014', '10001', '10003', '10010', '10018'],
 '10012': ['10013', '10003', '10014', '10278', '10002'],
 '10013': ['10278', '10007', '10012', '10282', '10038'],
 '10014': ['10011', '10012', '10003', '10013', '10001'],
 '10016': ['10010', '10017', '10154', '10112', '10020'],
 '10017': ['10154', '10022', '10112', '10016', '10020'],
 '10018': ['10036', '10001', '10020', '10112', '10019'],
 '10019': ['10036', '10020', '10112', '10023', '10018'],
 '10020': ['10112', '10154', '1

In [133]:
agg = df.groupby(["zip", "created_date"], as_index=False).sum()

all_dates = pd.date_range(df["created_date"].min(), df["created_date"].max())
all_zips = df["zip"].unique()

full_index = pd.MultiIndex.from_product([all_zips, all_dates], names=["zip", "created_date"])

agg = agg.set_index(["zip", "created_date"]).reindex(full_index, fill_value=0).reset_index()
agg = agg.sort_values(["zip", "created_date"])

agg["rolling_7d"] = agg.groupby("zip")["count"].transform(lambda x: x.rolling(7, min_periods=1).sum())

pivot = agg.pivot(index="created_date", columns="zip", values="count").fillna(0)
rolling_pivot = agg.pivot(index="created_date", columns="zip", values="rolling_7d").fillna(0)

agg[agg['zip']=="11102"].head(15)

,zip,created_date,count,rolling_7d
203763,11102,2020-01-01,0,0.0
203764,11102,2020-01-02,0,0.0
203765,11102,2020-01-03,0,0.0
203766,11102,2020-01-04,0,0.0
203767,11102,2020-01-05,0,0.0
203768,11102,2020-01-06,0,0.0
203769,11102,2020-01-07,0,0.0
203770,11102,2020-01-08,1,1.0
203771,11102,2020-01-09,0,1.0
203772,11102,2020-01-10,1,2.0


In [134]:
neighbor_lag7 = pd.DataFrame(0, index=pivot.index, columns=pivot.columns)

for z in pivot.columns:
    neighbors = neighbor_map.get(z, [])
    if neighbors:
        agg_neighbors = agg[agg['zip'].isin(neighbor_map.get(z, []))].sort_values('created_date')
neighbor_lag7[z] = (
    agg_neighbors
    .rolling('7D', on='created_date')       # 7-day rolling using actual dates
    .sum(numeric_only=True)['count']        # sum counts only
    .groupby(agg_neighbors['created_date']) # sum across all neighbor ZIPs per date
    .sum()
)

neighbor_lag14 = pd.DataFrame(0, index=pivot.index, columns=pivot.columns)

for z in pivot.columns:
    neighbors = neighbor_map.get(z, [])
    if neighbors:
        agg_neighbors = agg[agg['zip'].isin(neighbor_map.get(z, []))].sort_values('created_date')
neighbor_lag14[z] = (
    agg_neighbors
    .rolling('14D', on='created_date')       # 14-day rolling using actual dates
    .sum(numeric_only=True)['count']        # sum counts only
    .groupby(agg_neighbors['created_date']) # sum across all neighbor ZIPs per date
    .sum()
)


neighbor_lag30 = pd.DataFrame(0, index=pivot.index, columns=pivot.columns)

for z in pivot.columns:
    neighbors = neighbor_map.get(z, [])
    if neighbors:
        agg_neighbors = agg[agg['zip'].isin(neighbor_map.get(z, []))].sort_values('created_date')
neighbor_lag30[z] = (
    agg_neighbors
    .rolling('30D', on='created_date')       # 30-day rolling using actual dates
    .sum(numeric_only=True)['count']        # sum counts only
    .groupby(agg_neighbors['created_date']) # sum across all neighbor ZIPs per date
    .sum()
)

In [135]:
# neighbor_lag = pd.DataFrame(0, index=pivot.index, columns=pivot.columns)
# for z in pivot.columns:
#     neighbors = neighbor_map.get(z, [])
#     if neighbors:
#         neighbor_lag[z] = pivot[neighbors].rolling(7, min_periods=1).sum().sum(axis=1)



In [136]:
# Ensure datetime index
pivot.index = pd.to_datetime(pivot.index)

# (a) Weekend indicator
is_weekend = (pivot.index.weekday >= 5).astype(int)

# (b) Day of week (0=Mon, 6=Sun)
day_of_week = pivot.index.weekday

# Optional: one-hot encode day of week (recommended for RF)
dow_dummies = pd.get_dummies(day_of_week, prefix="dow")

# (c) Rolling average over the past year (365 days), lagged by 1 day
rolling_1y = pivot.rolling(window=365, min_periods=365).mean().shift(1)

rolling_6m = pivot.rolling(window=180, min_periods=180).mean().shift(1)

rolling_1m = pivot.rolling(window=30, min_periods=30).mean().shift(1)

rolling_2w = pivot.rolling(window=14, min_periods=14).mean().shift(1)

In [137]:
zip_models = {}
zip_results = defaultdict(dict)

for z in pivot.columns:
    X_zip = pd.DataFrame({
        "count": pivot[z][:-1],
        "rolling_7d": rolling_pivot[z][:-1],
        "neighbor_rolling_7d": neighbor_lag7[z][:-1],
        "neighbor_rolling_14d": neighbor_lag14[z][:-1],
        "neighbor_rolling_30d": neighbor_lag30[z][:-1],
        "rolling_1y": rolling_1y[z][:-1],
        "rolling_6m": rolling_6m[z][:-1],
        "rolling_1m": rolling_1m[z][:-1],
        "rolling_2w": rolling_2w[z][:-1],
        "is_weekend": is_weekend[:-1],
        "day_of_week": day_of_week[:-1],
    }, index=pivot.index[:-1])

    X_zip["delta_2w"] = pivot[z][:-1] - rolling_2w[z][:-1]
    X_zip["ratio_2w"] = pivot[z][:-1] / (rolling_2w[z][:-1] + 1)

    y_zip = pivot[z][1:]  # next-day target
    
    
    
    # Time-based train/test split (80% train, 20% test)

    split_idx = int(len(X_zip) * 0.8)
    X_train, X_test = X_zip[:split_idx], X_zip[split_idx:]
    y_train, y_test = y_zip[:split_idx], y_zip[split_idx:]

    # # Train Random Forest

    # Weight classes based on frequency
    # Count the frequencies of each class in y_train
    class_counts = Counter(y_train)
    total_samples = len(y_train)

    # Compute inverse proportional weights
    inv_weights = {cls: total_samples / count for cls, count in class_counts.items()}

    # Normalize to make them integers (optional: scale so the smallest weight is 1)

    class_weights_int = {0:1, 1: 50}
    
    rf = RandomForestClassifier(n_estimators = 500, # number of trees in ensemble
    max_depth = 10, # max_depth of each tree
    min_samples_leaf = 5, 
    max_features = 10,
    bootstrap= True, # sampling with replacement
    random_state = 216, # for consistency
    class_weight="balanced_subsample"
    )
    rf.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = rf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    zip_models[z] = rf
    zip_results[z]["accuracy"] = acc
    zip_results[z]["y_test"] = y_test
    zip_results[z]["y_pred"] = y_pred

In [ ]:
rows = []

# Loop over all ZIPs stored in pivot.columns
for i in range(len(pivot.columns)):
    zip_to_plot = pivot.columns[i]

    # Extract features and target
    X_zip = pd.DataFrame({
        "count": pivot[z][:-1],
        "rolling_7d": rolling_pivot[z][:-1],
        "neighbor_rolling_7d": neighbor_lag7[z][:-1],
        "neighbor_rolling_14d": neighbor_lag14[z][:-1],
        "neighbor_rolling_30d": neighbor_lag30[z][:-1],
        "rolling_1y": rolling_1y[z][:-1],
        "rolling_6m": rolling_6m[z][:-1],
        "rolling_1m": rolling_1m[z][:-1],
        "rolling_2w": rolling_2w[z][:-1],
        "is_weekend": is_weekend[:-1],
        "day_of_week": day_of_week[:-1],
    }, index=pivot.index[:-1])

    X_zip["delta_2w"] = pivot[z][:-1] - rolling_2w[z][:-1]
    X_zip["ratio_2w"] = pivot[z][:-1] / (rolling_2w[z][:-1] + 1)

    y_zip = pivot[zip_to_plot][1:]  # next-day counts

    split_idx = int(len(X_zip) * 0.8)
    X_train, X_test = X_zip[:split_idx], X_zip[split_idx:]
    y_train, y_test = y_zip[:split_idx], y_zip[split_idx:]

    # Predictions
    rf = zip_models[zip_to_plot]
    y_pred_train = rf.predict(X_train)
    y_pred_test = rf.predict(X_test)

    # Metrics
    rmse_train = np.sqrt(np.mean((y_train - y_pred_train) ** 2))
    rmse_test = np.sqrt(np.mean((y_test - y_pred_test) ** 2))
    rss_train = np.sum((y_train - y_pred_train) ** 2)
    rss_test = np.sum((y_test - y_pred_test) ** 2)
    precision, recall, _ = precision_recall_curve(y_test, y_pred_test)
    pr_auc = auc(recall, precision)
    
    # Store results
    rows.append({
        "zip": zip_to_plot,
        "accuracy": acc,
        "rmse_train": rmse_train,
        "rmse_test": rmse_test,
        "rss_train": rss_train,
        "rss_test": rss_test,
        "n_train": len(y_train),
        "n_test": len(y_test),
        "pr_auc": pr_auc,
    })

# Final table
results_table = pd.DataFrame(rows).set_index("zip")
display(results_table)

In [ ]:
results_table['pr_auc'].describe()

In [ ]:
# Loop over all ZIPs stored in pivot.columns
for i in range(len(pivot.columns)):
    zip_to_plot = pivot.columns[i]
    # Extract features and target
    X_zip = pd.DataFrame({
        "count": pivot[z][:-1],
        "rolling_7d": rolling_pivot[z][:-1],
        "neighbor_rolling_7d": neighbor_lag7[z][:-1],
        "neighbor_rolling_14d": neighbor_lag14[z][:-1],
        "rolling_1y": rolling_1y[z][:-1],
        "rolling_6m": rolling_6m[z][:-1],
        "rolling_1m": rolling_1m[z][:-1],
        "rolling_2w": rolling_2w[z][:-1],
        "is_weekend": is_weekend[:-1],
        "day_of_week": day_of_week[:-1],
    }, index=pivot.index[:-1])
    X_zip["delta_2w"] = pivot[z][:-1] - rolling_2w[z][:-1]
    X_zip["ratio_2w"] = pivot[z][:-1] / (rolling_2w[z][:-1] + 1)
    
    y_zip = pivot[zip_to_plot][1:]  # next-day counts

    split_idx = int(len(X_zip) * 0.8)
    X_train, X_test = X_zip[:split_idx], X_zip[split_idx:]
    y_train, y_test = y_zip[:split_idx], y_zip[split_idx:]

    # Get our predictions for train and test
    rf = zip_models[zip_to_plot]
    y_pred_train = rf.predict(X_train)
    y_pred_test = rf.predict(X_test)

    # Get our accuracy
    acc = zip_results[zip_to_plot]["accuracy"]

    # Compute RMSE and RSS on training and testing set
    rmse_train = np.sqrt(np.mean((y_train - y_pred_train)**2))
    rmse_test = np.sqrt(np.mean((y_test - y_pred_test)**2))
    rss_train = np.sum((y_train - y_pred_train)**2)
    rss_test = np.sum((y_test - y_pred_test)**2)

    # Plots
    
    plt.figure(figsize=(14, 6))
    dates_train = pivot.index[1:split_idx+1]  # aligned with next-day target
    dates_test = pivot.index[split_idx+1:]    

    # Training set
    plt.plot(dates_train, y_train, label=f"Actual (Train)", color="blue", alpha=0.5, linestyle='-')
    plt.plot(dates_train, y_pred_train, label=f"Predicted (Train) | RMSE: {rmse_train:.2f}, RSS: {rss_train:.0f}", color="green", linestyle='--')

    # Test set
    plt.plot(dates_test, y_test, label=f"Actual (Test)", color="red", alpha=0.5, linestyle='-')
    plt.plot(dates_test, y_pred_test, label=f"Predicted (Test) | RMSE: {rmse_test:.2f}, RSS: {rss_test:.0f}", color="orange", linestyle='--')

    plt.title(f"ZIP {zip_to_plot} - Accuracy: {acc:.3f}")
    plt.xlabel("Date")
    plt.ylabel("Number of Rats Spotted")
    plt.legend()
    plt.grid(True)
    plt.show()